In [3]:
import ROOT
import math
import os
from IPython.display import display

%jsroot on

# ============================================================
# SETTINGS
# ============================================================

file_path = "/root/geant4/detector/Tumor/Tumor1/tumor1.root"
tree_name = "t"

selected_volume = 2
selected_process = 2013
selected_pdg = 22

# Incident beam direction: /gps/direction 0 -1 0
incident_direction = (0.0, -1.0, 0.0)

# Set True for logarithmic count/color scale
use_logz = False


# ============================================================
# OPEN FILE
# ============================================================

if not os.path.exists(file_path):
    raise FileNotFoundError(file_path)

f = ROOT.TFile.Open(file_path)

if not f or f.IsZombie():
    raise RuntimeError(f"Could not open {file_path}")

t = f.Get(tree_name)

if not t:
    raise RuntimeError(f"Tree '{tree_name}' was not found")

print("Tree entries:", t.GetEntries())


# ============================================================
# REMOVE OLD OBJECTS
# ============================================================

for name in ["h_de_theta", "h_et_theta", "c_theta"]:
    obj = ROOT.gROOT.FindObject(name)
    if obj:
        obj.Delete()


# ============================================================
# HISTOGRAMS
# ============================================================

h_de_theta = ROOT.TH2F(
    "h_de_theta",
    "de versus scattering angle, vlm=2;"
    "#theta (degrees);de (keV)",
    180, 0, 180,
    140, 0, 700
)

h_et_theta = ROOT.TH2F(
    "h_et_theta",
    "et versus scattering angle, vlm=2;"
    "#theta (degrees);et (keV)",
    180, 0, 180,
    140, 0, 700
)

h_de_theta.SetStats(0)
h_et_theta.SetStats(0)


# ============================================================
# INCIDENT DIRECTION
# ============================================================

ix, iy, iz = incident_direction

incident_norm = math.sqrt(ix**2 + iy**2 + iz**2)

ix /= incident_norm
iy /= incident_norm
iz /= incident_norm


# ============================================================
# EVENT LOOP
# ============================================================

selected = 0

for event in t:

    n = min(
        len(event.pdg),
        len(event.pro),
        len(event.vlm),
        len(event.de),
        len(event.et),
        len(event.px),
        len(event.py),
        len(event.pz)
    )

    for i in range(n):

        if int(event.vlm[i]) != selected_volume:
            continue

        if int(event.pro[i]) != selected_process:
            continue

        if int(event.pdg[i]) != selected_pdg:
            continue

        # Do not require stp==0 initially.
        # Add this only after checking the stp distribution.
        #
        # if int(event.stp[i]) != 0:
        #     continue

        px = float(event.px[i])
        py = float(event.py[i])
        pz = float(event.pz[i])

        p = math.sqrt(px**2 + py**2 + pz**2)

        if p <= 0:
            continue

        ux = px / p
        uy = py / p
        uz = pz / p

        cos_theta = ix*ux + iy*uy + iz*uz
        cos_theta = max(-1.0, min(1.0, cos_theta))

        theta = math.degrees(math.acos(cos_theta))

        de_value = float(event.de[i])
        et_value = float(event.et[i])

        if math.isfinite(de_value):
            h_de_theta.Fill(theta, de_value)

        if math.isfinite(et_value):
            h_et_theta.Fill(theta, et_value)

        selected += 1


print("Selected points:", selected)
print("de entries:", h_de_theta.GetEntries())
print("et entries:", h_et_theta.GetEntries())

# ------------------------------------------------------------
# Bottom panel: et versus theta
# ------------------------------------------------------------

c.cd(2)

ROOT.gPad.SetLeftMargin(0.11)
ROOT.gPad.SetRightMargin(0.15)
ROOT.gPad.SetBottomMargin(0.12)
ROOT.gPad.SetTopMargin(0.09)

if h_et_theta.GetEntries() > 0:
    ROOT.gPad.SetLogz()

h_et_theta.Draw("COLZ")

# ============================================================
# 3D PLOT: angle vs et vs counts
# ============================================================

c3d = ROOT.TCanvas(
    "c3d",
    "Angle vs et vs counts",
    1100,
    800
)

c3d.SetRightMargin(0.12)
c3d.SetLeftMargin(0.10)
c3d.SetBottomMargin(0.10)

h_et_theta.SetTitle(
    "Compton events in volume 2;"
    "Scattering angle #theta (degrees);"
    "et (keV);"
    "Counts per bin"
)

# 3D colored surface
h_et_theta.Draw("LEGO2Z")

# Viewing angle
c3d.SetTheta(25)
c3d.SetPhi(35)

c3d.Modified()
c3d.Update()
c3d.Draw()

c3d

Tree entries: 100000
Selected points: 1397
de entries: 1397.0
et entries: 1397.0


NameError: name 'c' is not defined